# The Lab, as a notebook · Where did Sandy hit, and does the answer change?

Same three steps as `01_sandy_spatial.sql` and `02_sandy_spatial.py`, same numbers. The notebook adds what a worksheet can't:
**maps, charts, interactive controls, and Python loops over SQL**, all in one place, with the heavy lifting still running in Snowflake.

| Step | What we ask | What the notebook adds |
|---|---|---|
| 1 · Types | Which Manhattan ZIPs filed the most Sandy claims? | A ZIP map of Manhattan |
| 2 · Functions | Which counties near Manhattan, by **count** vs **mean paid**? | Twin county maps, a rank-flip chart, a live slider |
| 2b · Python | Does the distance we pick change the answer? | One loop, ten distances, one chart |
| 3 · Surprise | Were these properties even in the flood zone? | Inside vs outside the high-risk zone, by ZIP |

> Descriptive history for one event, from public FEMA data rounded for privacy. Not a flood risk model, and nothing here describes an individual property.

In [ ]:
# Parameters. Change a knob, then Run all.
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.use_warehouse("SDSC_WORKSHOP_WH")
# session.use_warehouse("SDSC_WORKSHOP_WH_LARGE")   # <-- scaling: 16x compute, one line

listing_db = "SNOWFLAKE_PUBLIC_DATA_FREE"   # <-- change if you renamed it
claims_tbl = f"{listing_db}.PUBLIC_DATA_FREE.FEMA_NATIONAL_FLOOD_INSURANCE_PROGRAM_CLAIM_INDEX"
geo_chars  = f"{listing_db}.PUBLIC_DATA_FREE.GEOGRAPHY_CHARACTERISTICS"
geo_index  = f"{listing_db}.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX"

distance_m = 25000   # <-- TRY: 75000
min_claims = 500     # <-- TRY: 5000

In [ ]:
# Chart helpers: Altair renders inline, and Snowflake GeoJSON becomes map shapes.
import json
import pandas as pd
import altair as alt
alt.renderers.enable("mimetype")

def to_pandas(r):
    return r.copy() if isinstance(r, pd.DataFrame) else r.to_pandas()

def for_d3(geom):
    # Snowflake emits counter-clockwise rings; Altair/d3 wants clockwise.
    if geom["type"] == "Polygon":
        geom["coordinates"] = [ring[::-1] for ring in geom["coordinates"]]
    elif geom["type"] == "MultiPolygon":
        geom["coordinates"] = [[ring[::-1] for ring in poly] for poly in geom["coordinates"]]
    return geom

def features(df, props):
    return alt.Data(values=[
        {"type": "Feature", "geometry": for_d3(json.loads(r["GEOJSON"])),
         "properties": {k: (r[k].item() if hasattr(r[k], "item") else r[k]) for k in props}}
        for _, r in df.iterrows()])

## Step 1 · Types: Manhattan by ZIP code

`TO_GEOGRAPHY` turns WKT text into a real shape; `ST_AREA`, `ST_CENTROID` and `ST_DISTANCE` measure it on the Earth's surface, in metres.
This time we also keep each shape as GeoJSON so we can draw it.

In [ ]:
%%sql -r zips
WITH manhattan_claims AS (
  SELECT zip_geo_id,
         COUNT(*)                                                     AS claims,
         ROUND(AVG(amount_paid_on_building_claim))                    AS mean_bldg_paid,
         COUNT_IF(NOT policy_rated_flood_zone ILIKE 'Special Flood%') AS outside_high_risk_zone
  FROM {{claims_tbl}}
  WHERE flood_event = 'Hurricane Sandy'
    AND county_geo_id = 'geoId/36061'
  GROUP BY zip_geo_id
),
zip_shapes AS (
  SELECT geo_id, TO_GEOGRAPHY(value) AS geog
  FROM {{geo_chars}}
  WHERE relationship_type = 'coordinates_wkt'
    AND geo_id IN (SELECT zip_geo_id FROM manhattan_claims)
),
southern_tip AS (
  SELECT ST_CENTROID(geog) AS pt FROM zip_shapes WHERE geo_id = 'zip/10004'
)
SELECT REPLACE(c.zip_geo_id, 'zip/', '')                       AS zip,
       c.claims,
       c.mean_bldg_paid,
       c.outside_high_risk_zone,
       ROUND(100 * c.outside_high_risk_zone / c.claims)         AS pct_outside,
       ROUND(ST_AREA(z.geog) / 1e6, 2)                          AS area_km2,
       ROUND(ST_DISTANCE(ST_CENTROID(z.geog), t.pt) / 1000, 1)  AS km_from_southern_tip,
       ST_ASGEOJSON(z.geog)::STRING                             AS geojson
FROM manhattan_claims c
JOIN zip_shapes z ON z.geo_id = c.zip_geo_id
CROSS JOIN southern_tip t
ORDER BY c.claims DESC

In [ ]:
z = to_pandas(zips)
zip_data = features(z, ["ZIP", "CLAIMS", "MEAN_BLDG_PAID", "PCT_OUTSIDE", "KM_FROM_SOUTHERN_TIP"])
zip_tip = [alt.Tooltip("properties.ZIP:N", title="ZIP"),
           alt.Tooltip("properties.CLAIMS:Q", title="Claims"),
           alt.Tooltip("properties.MEAN_BLDG_PAID:Q", title="Mean paid", format="$,.0f"),
           alt.Tooltip("properties.KM_FROM_SOUTHERN_TIP:Q", title="km from tip"),
           alt.Tooltip("properties.PCT_OUTSIDE:Q", title="% outside high-risk zone")]

zip_map = alt.Chart(zip_data).mark_geoshape(stroke="white", strokeWidth=0.6).encode(
    color=alt.Color("properties.CLAIMS:Q", title="Sandy claims", scale=alt.Scale(scheme="blues")),
    tooltip=zip_tip,
).project(type="mercator").properties(width=300, height=520, title="Manhattan: Sandy claims by ZIP")

top10 = alt.Chart(z.head(10)).mark_bar(color="#29B5E8").encode(
    x=alt.X("CLAIMS:Q", title="Claims"),
    y=alt.Y("ZIP:N", sort="-x", title=None),
    tooltip=["ZIP", "CLAIMS", "KM_FROM_SOUTHERN_TIP"],
).properties(width=260, height=300, title="Top 10 ZIPs")

zip_map | top10

**Read it:** the darkest ZIPs are not just at the southern tip. The East Village (10009) is #1, and East Harlem (10029), 12.5 km north, is #5. The surge pushed up both rivers. Hover a ZIP for the numbers.

## Step 2 · Functions: counties near Manhattan, ranked two ways

`TRY_TO_GEOGRAPHY` turns county WKT into polygons. `ST_DWITHIN` keeps counties within `distance_m` of Manhattan. Then we rank the same counties by **claim count** and by **mean building payout** (records with no payment are excluded from the mean and counted in `paid_missing`).

In [ ]:
%%sql -r counties
WITH county_claims AS (
  SELECT county_geo_id,
         COUNT(*)                                        AS claim_records,
         COUNT(amount_paid_on_building_claim)            AS paid_denominator,
         COUNT(*) - COUNT(amount_paid_on_building_claim) AS paid_missing,
         AVG(amount_paid_on_building_claim)              AS mean_bldg_paid_usd
  FROM {{claims_tbl}}
  WHERE flood_event   = 'Hurricane Sandy'
    AND state_geo_id IN ('geoId/36', 'geoId/34')
    AND county_geo_id IS NOT NULL
  GROUP BY county_geo_id
  HAVING COUNT(*) >= {{min_claims}}
),
county_shapes AS (
  SELECT geo_id, TRY_TO_GEOGRAPHY(value) AS boundary
  FROM {{geo_chars}}
  WHERE relationship_type = 'coordinates_wkt'
    AND geo_id IN (SELECT county_geo_id FROM county_claims UNION SELECT 'geoId/36061')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY geo_id
                             ORDER BY relationship_start_date DESC NULLS LAST) = 1
),
manhattan AS (SELECT boundary FROM county_shapes WHERE geo_id = 'geoId/36061'),
nearby AS (
  SELECT c.*, s.boundary,
         ROUND(ST_DISTANCE(s.boundary, m.boundary) / 1000, 1) AS km_from_manhattan
  FROM county_claims c
  JOIN county_shapes s ON s.geo_id = c.county_geo_id
  CROSS JOIN manhattan m
  WHERE ST_DWITHIN(s.boundary, m.boundary, {{distance_m}})
)
SELECT g.geo_name                                     AS county,
       claim_records,
       paid_missing,
       ROUND(mean_bldg_paid_usd)                      AS mean_bldg_paid_usd,
       km_from_manhattan,
       RANK() OVER (ORDER BY claim_records DESC)      AS rank_by_count,
       RANK() OVER (ORDER BY mean_bldg_paid_usd DESC) AS rank_by_mean_paid,
       ST_ASGEOJSON(ST_SIMPLIFY(boundary, 200))::STRING AS geojson
FROM nearby n
JOIN {{geo_index}} g ON g.geo_id = n.county_geo_id
ORDER BY rank_by_count

In [ ]:
c = to_pandas(counties)
county_data = features(c, ["COUNTY", "CLAIM_RECORDS", "MEAN_BLDG_PAID_USD", "PAID_MISSING",
                           "RANK_BY_COUNT", "RANK_BY_MEAN_PAID"])
county_tip = [alt.Tooltip("properties.COUNTY:N", title="County"),
              alt.Tooltip("properties.CLAIM_RECORDS:Q", title="Claims", format=","),
              alt.Tooltip("properties.MEAN_BLDG_PAID_USD:Q", title="Mean paid", format="$,.0f"),
              alt.Tooltip("properties.PAID_MISSING:Q", title="No payment recorded"),
              alt.Tooltip("properties.RANK_BY_COUNT:Q", title="Rank by count"),
              alt.Tooltip("properties.RANK_BY_MEAN_PAID:Q", title="Rank by mean")]

def county_map(field, title, scheme):
    return alt.Chart(county_data).mark_geoshape(stroke="white", strokeWidth=0.7).encode(
        color=alt.Color(f"properties.{field}:Q", title=title, scale=alt.Scale(scheme=scheme)),
        tooltip=county_tip,
    ).project(type="mercator").properties(width=360, height=420, title=title)

(county_map("CLAIM_RECORDS", "Claim count", "blues")
 | county_map("MEAN_BLDG_PAID_USD", "Mean building paid (USD)", "oranges")
).resolve_scale(color="independent")

In [ ]:
# Same counties, different question: each line is one county moving between rankings.
long = c.melt(id_vars=["COUNTY"], value_vars=["RANK_BY_COUNT", "RANK_BY_MEAN_PAID"],
              var_name="RANKING", value_name="RANK")
long["RANKING"] = long["RANKING"].map({"RANK_BY_COUNT": "By claim count",
                                       "RANK_BY_MEAN_PAID": "By mean paid"})
base = alt.Chart(long).encode(
    x=alt.X("RANKING:N", title=None, sort=["By claim count", "By mean paid"], axis=alt.Axis(labelAngle=0)),
    y=alt.Y("RANK:Q", scale=alt.Scale(reverse=True), title="Rank (1 = top)"),
    color=alt.Color("COUNTY:N", legend=None),
)
(base.mark_line(point=True)
 + base.mark_text(align="left", dx=6).encode(text="COUNTY:N")
       .transform_filter(alt.datum.RANKING == "By mean paid")
 + base.mark_text(align="right", dx=-6).encode(text="COUNTY:N")
       .transform_filter(alt.datum.RANKING == "By claim count")
).properties(width=420, height=380, title="The story flips")

**Read it:** Nassau leads by count. Manhattan is 9th by count but **1st by mean paid** ($236,483): fewer, much bigger buildings. And 223 of its 668 records have no payment, so always look at missingness.

**Who's missing?** Ocean County, NJ, the hardest-hit county of the whole event (37,459 claims). It's 60.6 km away. Set `distance_m = 75000` and Run all: it appears at #1.

### Step 2, live: drag the threshold

The query above filters with `min_claims` and reruns in Snowflake. Here we pull every NY/NJ county once (a few dozen rows), and let the slider re-rank them in the browser. Same idea as changing `min_claims`, without rerunning.

In [ ]:
all_counties = session.sql(f'''
    SELECT g.geo_name AS county,
           COUNT(*) AS claim_records,
           ROUND(AVG(c.amount_paid_on_building_claim)) AS mean_bldg_paid_usd
    FROM {claims_tbl} c
    JOIN {geo_index} g ON g.geo_id = c.county_geo_id
    WHERE c.flood_event = 'Hurricane Sandy'
      AND c.state_geo_id IN ('geoId/36', 'geoId/34')
    GROUP BY g.geo_name
    HAVING COUNT(*) >= 100
''').to_pandas()

threshold = alt.param(name="min_claims", value=500,
                      bind=alt.binding_range(min=100, max=10000, step=100, name="Minimum claims "))
alt.Chart(all_counties).mark_bar(color="#FF9F36").encode(
    x=alt.X("MEAN_BLDG_PAID_USD:Q", title="Mean building paid (USD)"),
    y=alt.Y("COUNTY:N", sort="-x", title=None),
    tooltip=["COUNTY", alt.Tooltip("CLAIM_RECORDS:Q", format=","),
             alt.Tooltip("MEAN_BLDG_PAID_USD:Q", format="$,.0f")],
).add_params(threshold).transform_filter(
    alt.datum.CLAIM_RECORDS >= threshold
).properties(width=420, title="Rank by mean paid, counties above the threshold")

**Try it:** slide to 5,000. Manhattan and Brooklyn disappear, and Queens takes the top spot. The threshold you pick changes which story you can tell.

## Step 2b · Python flex: does the distance we pick change the answer?

In SQL you change `distance_m` and rerun. In a notebook, one loop asks the question ten times. Each iteration is a full spatial query running in Snowflake; Python just collects the answers.

In [ ]:
sweep_sql = f'''
WITH county_claims AS (
  SELECT county_geo_id, COUNT(*) AS claim_records
  FROM {claims_tbl}
  WHERE flood_event = 'Hurricane Sandy'
    AND state_geo_id IN ('geoId/36', 'geoId/34') AND county_geo_id IS NOT NULL
  GROUP BY county_geo_id
),
county_shapes AS (
  SELECT geo_id, TRY_TO_GEOGRAPHY(value) AS boundary
  FROM {geo_chars}
  WHERE relationship_type = 'coordinates_wkt'
    AND geo_id IN (SELECT county_geo_id FROM county_claims)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY geo_id
                             ORDER BY relationship_start_date DESC NULLS LAST) = 1
)
SELECT COUNT(*) AS counties, SUM(c.claim_records) AS claims,
       MAX_BY(g.geo_name, c.claim_records) AS top_county
FROM county_claims c
JOIN county_shapes s ON s.geo_id = c.county_geo_id
JOIN {geo_index} g ON g.geo_id = c.county_geo_id
CROSS JOIN (SELECT boundary FROM county_shapes WHERE geo_id = 'geoId/36061') m
WHERE ST_DWITHIN(s.boundary, m.boundary, ?)
'''

rows = []
for km in [0, 10, 20, 25, 30, 40, 50, 60, 70, 80]:
    r = session.sql(sweep_sql, params=[km * 1000]).collect()[0]
    rows.append({"KM": km, "COUNTIES": r["COUNTIES"], "CLAIMS": r["CLAIMS"], "TOP_COUNTY": r["TOP_COUNTY"]})
sweep = pd.DataFrame(rows)
sweep

In [ ]:
line = alt.Chart(sweep).encode(
    x=alt.X("KM:Q", title="Distance from Manhattan (km)"),
    y=alt.Y("CLAIMS:Q", title="Sandy claims captured"),
    tooltip=["KM", "COUNTIES", alt.Tooltip("CLAIMS:Q", format=","), "TOP_COUNTY"],
)
(line.mark_line(color="#29B5E8") + line.mark_point(filled=True, size=70).encode(
    color=alt.Color("TOP_COUNTY:N", title="#1 county by claims"))
).properties(width=520, height=320, title="The circle you draw decides the answer")

**Read it:** the #1 county changes twice: Queens up to 10 km, Nassau from 20 km, then Ocean County from 70 km, when the circle reaches the Jersey Shore and the claim total jumps from ~73,000 to ~110,000. Same data, same SQL; only the question changed.

## Step 3 · The surprise: were these properties even in the flood zone?

`POLICY_RATED_FLOOD_ZONE` is the FEMA zone each policy was rated in at the time of the loss. "Special Flood..." is FEMA's high-risk zone (A and V). We already pulled it in Step 1.

In [ ]:
zone = z.head(10).assign(INSIDE=lambda d: d.CLAIMS - d.OUTSIDE_HIGH_RISK_ZONE)
zone_long = zone.melt(id_vars=["ZIP", "PCT_OUTSIDE"], value_vars=["INSIDE", "OUTSIDE_HIGH_RISK_ZONE"],
                      var_name="ZONE", value_name="N")
zone_long["ZONE"] = zone_long["ZONE"].map({"INSIDE": "Inside high-risk zone",
                                           "OUTSIDE_HIGH_RISK_ZONE": "Outside high-risk zone"})
bars = alt.Chart(zone_long).mark_bar().encode(
    x=alt.X("N:Q", title="Sandy claims"),
    y=alt.Y("ZIP:N", sort=list(zone.ZIP), title=None),
    color=alt.Color("ZONE:N", title=None,
                    scale=alt.Scale(range=["#29B5E8", "#FF9F36"])),
    tooltip=["ZIP", "ZONE", "N", alt.Tooltip("PCT_OUTSIDE:Q", title="% outside")],
).properties(width=380, height=300, title="Top 10 Manhattan ZIPs: where the claims came from")

outside_map = alt.Chart(zip_data).mark_geoshape(stroke="white", strokeWidth=0.6).encode(
    color=alt.Color("properties.PCT_OUTSIDE:Q", title="% outside zone", scale=alt.Scale(scheme="oranges")),
    tooltip=zip_tip,
).project(type="mercator").properties(width=300, height=520, title="% of claims outside the high-risk zone")

bars | outside_map

**Read it:** half of East Harlem's Sandy claims, and a third of the East Village's, came from properties FEMA had rated **outside** the high-risk zone. Across Manhattan it's 188 of 668 claims (28%), and they were paid more on average ($265,319 vs $227,798).

The flood map said these places were lower risk. The water disagreed. That's why underwriting needs geography.

### What you just did
SQL for the question, Python for the loop, and charts for the story, all in one notebook, with every spatial function running in Snowflake. Next: put it on a map others can explore (CARTO), and let business users ask it in plain English (CoWork).